# Models v4.1 — Long/Short + мягкая разметка (α=0.55)

**Два изменения относительно v4:**
1. **ALPHA = 0.55** (было 2) — мягче порог разметки → больше сигналов Buy/Sell (как в Wang et al.)
2. **ENABLE_SHORT = True** — добавлен шорт: модель может открывать короткие позиции по сигналу Sell

**Не менялось:** архитектура модели, fill_time_gaps, create_tensors (кроме ALPHA), INITIAL_EPOCHS=8, TEST_WEEKS=2, breakeven-логика, Kelly

In [ ]:
from tinkoff.invest import Client, AsyncClient, CandleInterval, SecurityTradingStatus, InstrumentStatus
from tinkoff.invest.services import InstrumentsService
from tinkoff.invest.utils import quotation_to_decimal, now
from tinkoff.invest.caching.instruments_cache.instruments_cache import InstrumentsCache

from tensorflow.keras import layers, models, optimizers
from tensorflow.keras import backend as K
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

import pyarrow
import pandas as pd
from pandas.tseries.offsets import DateOffset
from scipy import stats
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import os
import math
import time
from datetime import datetime, timezone, timedelta

from pathlib import Path
from typing import Dict, List, Tuple, Optional

from dotenv import load_dotenv

import warnings
warnings.filterwarnings("ignore")

In [ ]:
pd.set_option('display.max_columns', None) 
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 1000)

In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

In [ ]:
load_dotenv()
TOKEN = os.getenv("TOKEN")

## Конфигурация

Оригинальные параметры + единственное изменение `TEST_WEEKS` + параметры Kelly.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║                    КОНФИГУРАЦИЯ СТРАТЕГИИ v4.1                  ║
# ╚══════════════════════════════════════════════════════════════════╝

# --- Модель и обучение ---
TRAIN_MONTHS = 48
WINDOW_SIZE = 24
LEARNING_RATE = 0.0001
CONFIDENCE_THRESHOLD = 0.7
INITIAL_EPOCHS = 8          # Оптимум по val_loss curve
TEST_WEEKS = 2              # Недельный шаг переобучения

# --- Разметка (labeling) ---
ALPHA = 0.55                # ⬅ ИЗМЕНЕНИЕ: было 2, теперь 0.55 (как в Wang et al.)
                            # Мягкий порог → больше сигналов Buy/Sell

# --- Торговая стратегия ---
ENABLE_SHORT = True         # ⬅ ИЗМЕНЕНИЕ: добавлен шорт (как в Wang et al.)
                            # True = Long + Short, False = только Long (как было)

# --- Риск-менеджмент ---
COMMISSION = 0.0003
BE_TRIGGER = 1.0            # Перенос SL в безубыток при +1%
RF_ANNUAL = 0.15

# --- Критерий Келли ---
KELLY_FRACTION = 0.25
KELLY_MAX_POSITION = 0.30
KELLY_MIN_TRADES = 30
KELLY_LOOKBACK = 50

print("Конфигурация v4.1 загружена.")
print(f"  ИЗМЕНЕНИЕ 1: ALPHA={ALPHA} (было 2) — мягче порог, больше сигналов")
print(f"  ИЗМЕНЕНИЕ 2: ENABLE_SHORT={ENABLE_SHORT} — long + short торговля")
print(f"  Остальное без изменений: EPOCHS={INITIAL_EPOCHS}, TEST={TEST_WEEKS}нед, CONF={CONFIDENCE_THRESHOLD}")

## Предобработка данных (оригинал)

In [ ]:
def clean_market_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Очищает данные от пропусков и нулевых значений.
    ОРИГИНАЛЬНЫЙ КОД из models.ipynb
    """
    df = df.copy()
    
    # 1. Заменяем чистые нули на NaN (в цене Close/Open нулей быть не может)
    # Делаем это только для колонок с ценами
    cols_to_fix = ['Open', 'High', 'Low', 'Close']
    for col in cols_to_fix:
        if col in df.columns:
            df[col] = df[col].replace(0, pd.NA)
    
    # Forward + Backward fill для цен
    df[cols_to_fix] = df[cols_to_fix].ffill()
    df[cols_to_fix] = df[cols_to_fix].bfill()
    
    # Volume: пропуски = 0
    if 'Volume' in df.columns:
        df['Volume'] = df['Volume'].fillna(0)
    
    return df

In [ ]:
def fill_time_gaps(df: pd.DataFrame, interval_name: str = "5min") -> pd.DataFrame:
    """
    Вставляет пропущенные временные интервалы (строки), которых нет в данных.
    ОРИГИНАЛЬНЫЙ КОД из models.ipynb
    """
    resample_map = {
        "5min": "5min", "15min": "15min", "1hour": "h", "1day": "D"
    }
    freq = resample_map.get(interval_name, "5min")
    
    df = df.copy()
    
    if 'DateTime' in df.columns:
        df['DateTime'] = pd.to_datetime(df['DateTime'])
        df = df.set_index('DateTime')
    elif not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("DataFrame должен иметь колонку 'DateTime' или DatetimeIndex")
    
    df = df.sort_index()
    
    # Создаём полный диапазон
    full_range = pd.date_range(
        start=df.index.min(), 
        end=df.index.max(), 
        freq=freq
    )
    
    df = df.reindex(full_range)
    df.index.name = "DateTime"
    
    # Заполнение
    price_cols = ['Open', 'High', 'Low', 'Close']
    df[price_cols] = df[price_cols].ffill().bfill()
    if 'Volume' in df.columns:
        df['Volume'] = df['Volume'].fillna(0)
    
    return df.reset_index()

## Модель (оригинал)

In [ ]:
def build_model(input_shape):
    """ОРИГИНАЛЬНАЯ архитектура из models.ipynb"""
    model = models.Sequential([
        layers.Conv1D(64, kernel_size=3, padding='same', activation='relu', input_shape=input_shape),
        layers.MaxPooling1D(2),
        layers.Conv1D(128, kernel_size=3, padding='same', activation='relu'),
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(3, activation='softmax') 
    ])
    
    # Использование legacy оптимизатора для ускорения на M1/M2/M3
    try:
        opt = optimizers.legacy.Adam(learning_rate=LEARNING_RATE)
        print("Используется оптимизатор legacy.Adam для Mac")
    except AttributeError:
        opt = optimizers.Adam(learning_rate=LEARNING_RATE)
        print("Используется стандартный Adam")

    model.compile(
        optimizer=opt, 
        loss='sparse_categorical_crossentropy', 
        metrics=['accuracy']
    )
    return model

In [ ]:
def create_tensors(df, window_size=WINDOW_SIZE, alpha=ALPHA):
    """
    Создает X (тензоры для CNN) и y (таргеты по волатильности).
    ОРИГИНАЛЬНЫЙ КОД из models.ipynb
    """
    df = df.copy().sort_values("DateTime")
    
    # Расчет таргета
    returns = df['Close'].pct_change()
    volatility = returns.rolling(window=10).std()
    # Изменение цены через 24 свечи (2 часа для 5-мин интервала)
    future_change = (df['Close'].shift(-window_size) - df['Close']) / df['Close']
    
    conditions = [
        (future_change >= alpha * volatility), # Up (1)
        (future_change <= -alpha * volatility) # Down (2)
    ]
    df['Target'] = np.select(conditions, [1, 2], default=0) # Flat (0)
    
    # Фичи: OHLCV
    features = ['Open', 'High', 'Low', 'Close', 'Volume']
    data_x = df[features].values
    data_y = df['Target'].values
    dates = df['DateTime'].values
    prices = df['Close'].values

    X, y, out_dates, out_prices = [], [], [], []

    for i in range(window_size, len(df) - window_size):
        window = data_x[i-window_size:i]
        
        # Локальная нормализация окна (Z-score) для стабильности CNN
        scaler = StandardScaler()
        norm_window = scaler.fit_transform(window)
        
        X.append(norm_window)
        y.append(data_y[i])
        out_dates.append(dates[i])
        out_prices.append(prices[i])

    return np.array(X), np.array(y), np.array(out_dates), np.array(out_prices)

## Бэктест

**Единственное отличие от оригинала:** шаг `TEST_WEEKS` (2 недели) вместо `TEST_MONTHS` (1 месяц). Вся остальная логика — точная копия.

In [ ]:
def run_backtest_with_fine_tuning(df, confidence_threshold=CONFIDENCE_THRESHOLD):
    """
    Оригинальный бэктест из models.ipynb.
    Единственное изменение: шаг = TEST_WEEKS недель вместо TEST_MONTHS месяцев.
    """
    total_months = (df['DateTime'].max() - df['DateTime'].min()).days // 30
    if total_months < TRAIN_MONTHS + 1:
        print(f"ВНИМАНИЕ: Недостаточно данных! Всего месяцев: {total_months}, нужно минимум: {TRAIN_MONTHS + 1}")
    else:
        test_weeks_est = (total_months - TRAIN_MONTHS) * 4.3
        print(f"Данных достаточно. Обучение: {TRAIN_MONTHS} мес. "
              f"Тест: ~{test_weeks_est:.0f} нед. (шаг {TEST_WEEKS} нед.)")
        
    # Оригинальный create_tensors
    X_all, y_all, dates_all, prices_all = create_tensors(df, alpha=ALPHA) 
    dt_index = pd.to_datetime(dates_all)
    
    start_date = dt_index.min()
    current_test_start = start_date + DateOffset(months=TRAIN_MONTHS)
    
    model = build_model(input_shape=(WINDOW_SIZE, 5))
    
    # Оригинальное первичное обучение
    initial_mask = (dt_index >= start_date) & (dt_index < current_test_start)
    model.fit(X_all[initial_mask], y_all[initial_mask], epochs=15, batch_size=32, verbose=1)
    
    all_preds = []
    step = 0

    while current_test_start < dt_index.max():
        # ⬅ ЕДИНСТВЕННОЕ ИЗМЕНЕНИЕ: Timedelta(weeks=) вместо DateOffset(months=)
        test_end = current_test_start + pd.Timedelta(weeks=TEST_WEEKS)
        test_mask = (dt_index >= current_test_start) & (dt_index < test_end)
        
        X_test, y_test = X_all[test_mask], y_all[test_mask]
        if len(X_test) == 0:
            current_test_start = test_end
            continue
            
        probs = model.predict(X_test, verbose=0)
        
        # Оригинальная логика фильтрации
        final_classes = []
        for p in probs:
            if p[1] > CONFIDENCE_THRESHOLD:
                final_classes.append(1)
            elif p[2] > CONFIDENCE_THRESHOLD:
                final_classes.append(2)
            else:
                final_classes.append(0)
        
        chunk_results = pd.DataFrame({
            'DateTime': dates_all[test_mask],
            'Price': prices_all[test_mask],
            'Actual': y_test,
            'Predicted': final_classes
        })
        all_preds.append(chunk_results)
        
        # Оригинальное дообучение: 1 эпоха, тот же LR
        model.fit(X_test, y_test, epochs=1, batch_size=32, verbose=0)
        
        step += 1
        current_test_start = test_end

    result = pd.concat(all_preds)
    print(f"\nБэктест завершён. Шагов: {step}, предсказаний: {len(result)}")
    return result

## Критерий Келли — расширенная реализация

Три формулировки по статье [mlgu.ru/3740](https://mlgu.ru/3740/):

**1. Дискретный Келли** (классический, для трейдинга с дискретными сделками):
$$f^* = p - \frac{1-p}{b}$$
где $p$ — win rate, $b$ = avg_win / avg_loss (payoff ratio).

**2. Непрерывный Келли** (для нормально распределённых доходностей):
$$f^* = \frac{\mu}{\sigma^2}$$

**3. Непрерывный с безрисковой ставкой** (критично при RF_ANNUAL = 15%):
$$f^* = \frac{\mu - r_f}{\sigma^2}$$

Все три варианта дополнены фракционным коэффициентом $c$ и ограничением максимальной позиции.

Также реализованы:
- Скользящий Келли по последним N сделкам
- Анализ чувствительности (влияние ошибок в p и r на f*)
- Сравнение equity curve при Full / Half / Quarter Kelly

In [ ]:
class KellyCriterion:
    """
    Критерий Келли, адаптированный для алгоритмической торговли.
    
    Реализует три формулировки:
        1. Дискретный: f* = p - (1-p)/b
        2. Непрерывный: f* = μ/σ²
        3. Непрерывный с rf: f* = (μ - rf) / σ²
    
    Источник: https://mlgu.ru/3740/
    """
    
    def __init__(self,
                 fraction: float = KELLY_FRACTION,
                 max_position: float = KELLY_MAX_POSITION,
                 min_trades: int = KELLY_MIN_TRADES,
                 lookback: int = KELLY_LOOKBACK,
                 commission: float = COMMISSION,
                 rf_annual: float = RF_ANNUAL):
        self.fraction = fraction
        self.max_position = max_position
        self.min_trades = min_trades
        self.lookback = lookback
        self.commission = commission
        self.rf_annual = rf_annual
        self.history: List[Dict] = []
    
    # ────────────────────────────────────────────────
    # 1. Дискретный Келли: f* = p - (1-p)/b
    # ────────────────────────────────────────────────
    def discrete_kelly(self, trades_pnl: np.ndarray) -> Dict:
        """
        Классический дискретный Келли по win rate и payoff ratio.
        Формула: f* = p - (1-p) / b
        """
        n = len(trades_pnl)
        result = self._empty_result('discrete')
        result['n_trades'] = n
        
        if n < self.min_trades:
            return result
        
        net_pnl = trades_pnl - (self.commission * 2)
        wins = net_pnl[net_pnl > 0]
        losses = net_pnl[net_pnl < 0]
        
        if len(wins) == 0 or len(losses) == 0:
            return result
        
        p = len(wins) / n
        q = 1 - p
        avg_win = wins.mean()
        avg_loss = abs(losses.mean())
        
        if avg_loss == 0:
            return result
        
        b = avg_win / avg_loss
        
        # f* = p - q/b  (эквивалентно (p*b - q) / b)
        kelly_full = p - q / b
        
        result.update({
            'kelly_full': round(kelly_full, 6),
            'kelly_fractional': round(kelly_full * self.fraction, 6),
            'kelly_capped': round(np.clip(kelly_full * self.fraction, 0.0, self.max_position), 6),
            'win_rate': round(p, 4),
            'avg_win': round(avg_win, 6),
            'avg_loss': round(avg_loss, 6),
            'payoff_ratio': round(b, 4),
            'is_valid': kelly_full > 0,
            'edge': round(p * avg_win - q * avg_loss, 6),
        })
        
        return result
    
    # ────────────────────────────────────────────────
    # 2. Непрерывный Келли: f* = μ / σ²
    # ────────────────────────────────────────────────
    def continuous_kelly(self, trades_pnl: np.ndarray) -> Dict:
        """
        Непрерывный Келли для нормально распределённых доходностей.
        Формула: f* = μ / σ²
        """
        n = len(trades_pnl)
        result = self._empty_result('continuous')
        result['n_trades'] = n
        
        if n < self.min_trades:
            return result
        
        net_pnl = trades_pnl - (self.commission * 2)
        mu = net_pnl.mean()
        sigma2 = net_pnl.var()
        
        if sigma2 == 0:
            return result
        
        kelly_full = mu / sigma2
        
        result.update({
            'kelly_full': round(kelly_full, 6),
            'kelly_fractional': round(kelly_full * self.fraction, 6),
            'kelly_capped': round(np.clip(kelly_full * self.fraction, 0.0, self.max_position), 6),
            'mu': round(mu, 6),
            'sigma2': round(sigma2, 8),
            'sigma': round(np.sqrt(sigma2), 6),
            'is_valid': kelly_full > 0,
        })
        
        return result
    
    # ────────────────────────────────────────────────
    # 3. Непрерывный с безрисковой ставкой: f* = (μ - rf) / σ²
    # ────────────────────────────────────────────────
    def continuous_kelly_rf(self, trades_pnl: np.ndarray, avg_holding_days: float = 1.0) -> Dict:
        """
        Непрерывный Келли с учётом безрисковой ставки.
        Формула: f* = (μ - rf_per_trade) / σ²
        
        rf пересчитывается на средний период удержания сделки.
        """
        n = len(trades_pnl)
        result = self._empty_result('continuous_rf')
        result['n_trades'] = n
        
        if n < self.min_trades:
            return result
        
        net_pnl = trades_pnl - (self.commission * 2)
        mu = net_pnl.mean()
        sigma2 = net_pnl.var()
        
        if sigma2 == 0:
            return result
        
        # Пересчёт годовой безрисковой ставки на период одной сделки
        rf_per_trade = self.rf_annual * (avg_holding_days / 365.25)
        
        kelly_full = (mu - rf_per_trade) / sigma2
        
        result.update({
            'kelly_full': round(kelly_full, 6),
            'kelly_fractional': round(kelly_full * self.fraction, 6),
            'kelly_capped': round(np.clip(kelly_full * self.fraction, 0.0, self.max_position), 6),
            'mu': round(mu, 6),
            'rf_per_trade': round(rf_per_trade, 8),
            'excess_return': round(mu - rf_per_trade, 6),
            'sigma2': round(sigma2, 8),
            'is_valid': kelly_full > 0,
        })
        
        return result
    
    # ────────────────────────────────────────────────
    # Скользящий Келли
    # ────────────────────────────────────────────────
    def rolling_kelly(self, trades_pnl: np.ndarray, method: str = 'discrete') -> pd.DataFrame:
        """
        Скользящий расчёт Келли по последним `lookback` сделкам.
        method: 'discrete', 'continuous', 'continuous_rf'
        """
        records = []
        calc_fn = {
            'discrete': self.discrete_kelly,
            'continuous': self.continuous_kelly,
            'continuous_rf': lambda x: self.continuous_kelly_rf(x, avg_holding_days=5.0),
        }[method]
        
        for i in range(self.min_trades, len(trades_pnl)):
            start = max(0, i - self.lookback)
            window = trades_pnl[start:i]
            result = calc_fn(window)
            result['trade_index'] = i
            records.append(result)
        
        return pd.DataFrame(records)
    
    # ────────────────────────────────────────────────
    # Анализ чувствительности
    # ────────────────────────────────────────────────
    def sensitivity_analysis(self, trades_pnl: np.ndarray):
        """
        Визуализация чувствительности f* к ошибкам в оценке p и r.
        По мотивам статьи mlgu.ru/3740.
        """
        net_pnl = trades_pnl - (self.commission * 2)
        wins = net_pnl[net_pnl > 0]
        losses = net_pnl[net_pnl < 0]
        
        if len(wins) == 0 or len(losses) == 0:
            print("Недостаточно данных для анализа чувствительности.")
            return
        
        true_p = len(wins) / len(net_pnl)
        true_r = wins.mean() / abs(losses.mean())
        true_kelly = true_p - (1 - true_p) / true_r
        
        # Сетка параметров
        p_range = np.linspace(max(0.01, true_p - 0.15), min(0.99, true_p + 0.15), 80)
        r_range = np.linspace(max(0.1, true_r - 0.8), true_r + 0.8, 80)
        p_grid, r_grid = np.meshgrid(p_range, r_range)
        kelly_grid = p_grid - (1 - p_grid) / r_grid
        
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        
        # 1. Контурный график f*
        contour = axes[0].contourf(p_grid, r_grid, kelly_grid * 100, 20, cmap='RdYlGn')
        axes[0].plot(true_p, true_r, 'k*', markersize=15, label=f'Текущее (p={true_p:.2f}, r={true_r:.2f})')
        axes[0].set_xlabel('Win Rate (p)')
        axes[0].set_ylabel('Payoff Ratio (r)')
        axes[0].set_title('f* по Келли (%)')
        axes[0].legend(fontsize=9)
        plt.colorbar(contour, ax=axes[0])
        
        # 2. Срез по p при фиксированном r
        idx_r = np.abs(r_range - true_r).argmin()
        axes[1].plot(p_range, kelly_grid[idx_r, :] * 100, 'b-', linewidth=2)
        axes[1].axhline(y=true_kelly * 100, color='r', linestyle='--', alpha=0.7, label=f'f*={true_kelly*100:.1f}%')
        axes[1].axvline(x=true_p, color='r', linestyle='--', alpha=0.7)
        axes[1].axhline(y=0, color='k', linestyle='-', alpha=0.3)
        axes[1].fill_between(p_range, kelly_grid[idx_r, :] * 100 * self.fraction, alpha=0.2, color='green',
                             label=f'Fractional (×{self.fraction})')
        axes[1].set_xlabel('Win Rate (p)')
        axes[1].set_ylabel('f* (%)')
        axes[1].set_title(f'Чувствительность к p (r={true_r:.2f})')
        axes[1].legend(fontsize=9)
        axes[1].grid(True, alpha=0.3)
        
        # 3. Срез по r при фиксированном p
        idx_p = np.abs(p_range - true_p).argmin()
        axes[2].plot(r_range, kelly_grid[:, idx_p] * 100, 'b-', linewidth=2)
        axes[2].axhline(y=true_kelly * 100, color='r', linestyle='--', alpha=0.7, label=f'f*={true_kelly*100:.1f}%')
        axes[2].axvline(x=true_r, color='r', linestyle='--', alpha=0.7)
        axes[2].axhline(y=0, color='k', linestyle='-', alpha=0.3)
        axes[2].fill_between(r_range, kelly_grid[:, idx_p] * 100 * self.fraction, alpha=0.2, color='green',
                             label=f'Fractional (×{self.fraction})')
        axes[2].set_xlabel('Payoff Ratio (r)')
        axes[2].set_ylabel('f* (%)')
        axes[2].set_title(f'Чувствительность к r (p={true_p:.2f})')
        axes[2].legend(fontsize=9)
        axes[2].grid(True, alpha=0.3)
        
        plt.suptitle('Анализ чувствительности критерия Келли', fontsize=14, y=1.02)
        plt.tight_layout()
        plt.show()
        
        print(f"Текущие параметры: p={true_p:.4f}, r={true_r:.4f}")
        print(f"Kelly (полный):    {true_kelly*100:.2f}%")
        print(f"Kelly (×{self.fraction}):    {true_kelly*self.fraction*100:.2f}%")
    
    # ────────────────────────────────────────────────
    # Сравнение equity curves (Full / Half / Quarter)
    # ────────────────────────────────────────────────
    def compare_fractions(self, trades_pnl: np.ndarray, initial_capital: float = 100000):
        """
        Сравнение динамики капитала при разных фракциях Келли.
        По мотивам статьи mlgu.ru/3740.
        """
        discrete = self.discrete_kelly(trades_pnl)
        if not discrete['is_valid']:
            print("Kelly невалиден (нет edge). Сравнение невозможно.")
            return
        
        kelly_f = discrete['kelly_full']
        net_pnl = trades_pnl - (self.commission * 2)
        
        strategies = {
            f'Full Kelly ({kelly_f*100:.1f}%)': kelly_f,
            f'Half Kelly ({kelly_f*50:.1f}%)': kelly_f * 0.5,
            f'Quarter Kelly ({kelly_f*25:.1f}%)': kelly_f * 0.25,
            'Fixed 1%': 0.01,
        }
        
        results = {}
        metrics = {}
        
        for name, frac in strategies.items():
            capital = initial_capital
            capitals = [capital]
            max_capital = capital
            max_dd = 0
            
            for ret in net_pnl:
                position_size = np.clip(frac, 0, 1.0)
                capital *= (1 + ret * position_size)
                capitals.append(capital)
                max_capital = max(max_capital, capital)
                dd = (max_capital - capital) / max_capital
                max_dd = max(max_dd, dd)
            
            results[name] = capitals
            total_ret = (capitals[-1] / capitals[0] - 1) * 100
            
            rets = np.diff(capitals) / np.array(capitals[:-1])
            vol = np.std(rets) * np.sqrt(len(rets)) if len(rets) > 0 else 0
            
            metrics[name] = {
                'Итого (%)': round(total_ret, 2),
                'Max DD (%)': round(max_dd * 100, 2),
                'Vol (%)': round(vol * 100, 2),
            }
        
        # Визуализация
        fig, ax = plt.subplots(figsize=(14, 6))
        for name, capitals in results.items():
            ax.plot(capitals, label=name, linewidth=2)
        ax.set_title('Сравнение фракций Келли: динамика капитала')
        ax.set_xlabel('Номер сделки')
        ax.set_ylabel(f'Капитал (начало = {initial_capital:,.0f})')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        # Таблица метрик
        metrics_df = pd.DataFrame(metrics).T
        print("\nМетрики по фракциям Келли:")
        print(metrics_df.to_string())
        
        return metrics_df
    
    # ────────────────────────────────────────────────
    # Полный отчёт (все три формулировки)
    # ────────────────────────────────────────────────
    def full_report(self, trades_pnl: np.ndarray, avg_holding_days: float = 1.0, label: str = ""):
        """Сводный отчёт по всем трём формулировкам Келли."""
        d = self.discrete_kelly(trades_pnl)
        c = self.continuous_kelly(trades_pnl)
        cr = self.continuous_kelly_rf(trades_pnl, avg_holding_days)
        
        print(f"\n{'═'*70}")
        print(f"КРИТЕРИЙ КЕЛЛИ — ПОЛНЫЙ ОТЧЁТ{' — ' + label if label else ''}")
        print(f"{'═'*70}")
        print(f"Сделок в выборке:          {d['n_trades']}")
        print(f"")
        print(f"{'─'*70}")
        print(f"1. ДИСКРЕТНЫЙ: f* = p - (1-p)/b")
        print(f"{'─'*70}")
        print(f"   Win Rate (p):           {d.get('win_rate',0)*100:.1f}%")
        print(f"   Avg Win:                {d.get('avg_win',0)*100:.3f}%")
        print(f"   Avg Loss:               {d.get('avg_loss',0)*100:.3f}%")
        print(f"   Payoff Ratio (b):       {d.get('payoff_ratio',0):.2f}")
        print(f"   Edge:                   {d.get('edge',0)*100:.3f}%")
        print(f"   Kelly (полный):         {d['kelly_full']*100:.2f}%")
        print(f"   Kelly (×{self.fraction}):          {d['kelly_fractional']*100:.2f}%")
        print(f"   Kelly (capped):         {d['kelly_capped']*100:.2f}%")
        print(f"   Валидность:             {'✓' if d['is_valid'] else '✗'}")
        print(f"")
        print(f"{'─'*70}")
        print(f"2. НЕПРЕРЫВНЫЙ: f* = μ/σ²")
        print(f"{'─'*70}")
        print(f"   μ (ср. доходность):     {c.get('mu',0)*100:.4f}%")
        print(f"   σ (волатильность):      {c.get('sigma',0)*100:.4f}%")
        print(f"   Kelly (полный):         {c['kelly_full']*100:.2f}%")
        print(f"   Kelly (×{self.fraction}):          {c['kelly_fractional']*100:.2f}%")
        print(f"   Kelly (capped):         {c['kelly_capped']*100:.2f}%")
        print(f"   Валидность:             {'✓' if c['is_valid'] else '✗'}")
        print(f"")
        print(f"{'─'*70}")
        print(f"3. НЕПРЕРЫВНЫЙ с rf: f* = (μ - rf) / σ²")
        print(f"{'─'*70}")
        print(f"   rf (годовая):           {self.rf_annual*100:.1f}%")
        print(f"   rf (на сделку):         {cr.get('rf_per_trade',0)*100:.5f}%")
        print(f"   μ - rf:                 {cr.get('excess_return',0)*100:.4f}%")
        print(f"   Kelly (полный):         {cr['kelly_full']*100:.2f}%")
        print(f"   Kelly (×{self.fraction}):          {cr['kelly_fractional']*100:.2f}%")
        print(f"   Kelly (capped):         {cr['kelly_capped']*100:.2f}%")
        print(f"   Валидность:             {'✓' if cr['is_valid'] else '✗'}")
        print(f"{'═'*70}")
        
        self.history.append({'discrete': d, 'continuous': c, 'continuous_rf': cr})
        return d, c, cr
    
    # ────────────────────────────────────────────────
    # Утилиты
    # ────────────────────────────────────────────────
    def _empty_result(self, method: str = 'discrete') -> Dict:
        return {
            'method': method,
            'n_trades': 0,
            'kelly_full': 0.0,
            'kelly_fractional': 0.0,
            'kelly_capped': 0.0,
            'is_valid': False,
        }

## Анализ стратегии

In [ ]:
def analyze_strategy_performance(df, commission=COMMISSION, rf_rate=RF_ANNUAL, be_trigger=BE_TRIGGER):
    """
    Анализ стратегии с поддержкой Long/Short и breakeven-логикой.
    
    Логика (как в Wang et al.):
        - Signal 1 (Buy):  если нет позиции → открыть Long
                           если в Short → закрыть Short, открыть Long
        - Signal 2 (Sell): если нет позиции и ENABLE_SHORT → открыть Short
                           если в Long → закрыть Long, открыть Short (если ENABLE_SHORT)
        - Signal 0 (Hold): ничего не делать
    
    Breakeven работает для обоих направлений:
        - Long:  если цена выросла на BE_TRIGGER%, SL переносится на entry
        - Short: если цена упала на BE_TRIGGER%, SL переносится на entry
    """
    trades = []
    position = 0       # 0 = нет, 1 = long, -1 = short
    entry_p, entry_d = 0.0, None
    is_breakeven = False
    
    def close_position(price, date, exit_type='Signal'):
        """Закрывает текущую позицию и добавляет сделку."""
        nonlocal position, entry_p, entry_d, is_breakeven
        
        if position == 1:  # Long
            pnl = (price - entry_p) / entry_p
        else:              # Short
            pnl = (entry_p - price) / entry_p
        
        pnl_net = (pnl - commission * 2) * 100
        
        if exit_type == 'Breakeven':
            pnl_net = -(commission * 2) * 100  # Только комиссия
        
        duration = date - entry_d
        trades.append({
            'Entry Date': entry_d,
            'Exit Date': date,
            'Entry Price': round(entry_p, 2),
            'Exit Price': round(price, 2),
            'Profit %': round(pnl_net, 4),
            'Profit (доля)': round(pnl_net / 100, 6),
            'Type': exit_type,
            'Direction': 'Long' if position == 1 else 'Short',
            'Duration Days': duration.days,
            'Duration Months': round(duration.days / 30.4375, 2)
        })
        position = 0
        is_breakeven = False
    
    for i in range(len(df)):
        sig = df['Predicted'].iloc[i]
        price = df['Price'].iloc[i]
        date = df['DateTime'].iloc[i]
        
        # ── Проверка breakeven для открытой позиции ──
        if position != 0:
            if position == 1:  # Long
                current_profit_pct = (price - entry_p) / entry_p * 100
            else:              # Short
                current_profit_pct = (entry_p - price) / entry_p * 100
            
            # Активация breakeven
            if not is_breakeven and current_profit_pct >= be_trigger:
                is_breakeven = True
            
            # Срабатывание breakeven (цена вернулась к entry)
            if is_breakeven and current_profit_pct <= 0:
                close_position(price, date, 'Breakeven')
                continue
        
        # ── Сигнал Buy (1) ──
        if sig == 1:
            if position == -1:   # В шорте → закрыть шорт
                close_position(price, date, 'Signal')
            if position == 0:    # Открыть лонг
                position = 1
                entry_p = price
                entry_d = date
                is_breakeven = False
        
        # ── Сигнал Sell (2) ──
        elif sig == 2:
            if position == 1:    # В лонге → закрыть лонг
                close_position(price, date, 'Signal')
            if position == 0 and ENABLE_SHORT:  # Открыть шорт
                position = -1
                entry_p = price
                entry_d = date
                is_breakeven = False
        
        # ── Конец данных — закрыть всё ──
        if i == len(df) - 1 and position != 0:
            close_position(price, date, 'Signal')
    
    t_df = pd.DataFrame(trades)
    
    if t_df.empty:
        print("Сделок не найдено.")
        return None, None
    
    # ── Метрики ──
    pnls_pct = t_df['Profit %']
    pnls = t_df['Profit (доля)']
    n = len(pnls)
    
    n_long = len(t_df[t_df['Direction'] == 'Long'])
    n_short = len(t_df[t_df['Direction'] == 'Short'])
    n_be = len(t_df[t_df['Type'] == 'Breakeven'])
    n_sig = len(t_df[t_df['Type'] == 'Signal'])
    
    win_rate = (pnls > 0).mean() * 100
    total_ret = pnls_pct.sum()
    avg_pnl = pnls_pct.mean()
    
    gross_profit = pnls_pct[pnls_pct > 0].sum()
    gross_loss = abs(pnls_pct[pnls_pct < 0].sum())
    pf = gross_profit / gross_loss if gross_loss > 0 else np.inf
    
    avg_win = pnls_pct[pnls_pct > 0].mean() if (pnls_pct > 0).any() else 0
    avg_loss_val = abs(pnls_pct[pnls_pct < 0].mean()) if (pnls_pct < 0).any() else 0
    rr = avg_win / avg_loss_val if avg_loss_val > 0 else np.inf
    
    std_pnl = pnls_pct.std()
    sharpe = ((pnls_pct.mean() - (rf_rate / 252 * 100)) / std_pnl * np.sqrt(n)) if std_pnl > 0 and n > 1 else 0
    mde = 2.8 * (std_pnl / np.sqrt(n)) if n > 1 else 0
    stat_sig = abs(avg_pnl) > mde
    
    # Метрики по направлениям
    long_trades = t_df[t_df['Direction'] == 'Long']['Profit %']
    short_trades = t_df[t_df['Direction'] == 'Short']['Profit %']
    
    # ── Келли ──
    kelly = KellyCriterion()
    kelly_result = kelly.discrete_kelly(pnls.values)
    
    # ── Вывод ──
    print(f"\n{'═'*70}")
    print(f"ИТОГО ПО СТРАТЕГИИ ({'Long+Short' if ENABLE_SHORT else 'Long only'} + breakeven + Kelly)")
    print(f"{'═'*70}")
    print(f"Всего сделок:              {n}")
    print(f"  Long:                    {n_long}   (ср. {long_trades.mean():.2f}%)" if n_long > 0 else f"  Long:                    0")
    print(f"  Short:                   {n_short}   (ср. {short_trades.mean():.2f}%)" if n_short > 0 else f"  Short:                   0")
    print(f"  Breakeven:               {n_be}")
    print(f"  Signal:                  {n_sig}")
    print(f"Win Rate:                  {win_rate:.2f}%")
    print(f"Общая доходность:          {total_ret:.2f}%")
    print(f"Средняя сделка:            {avg_pnl:.2f}%")
    print(f"Profit Factor:             {pf:.2f}")
    print(f"Avg RR Ratio:              {rr:.2f}")
    print(f"Sharpe Ratio:              {sharpe:.2f}")
    print(f"{'─'*70}")
    print(f"MDE (стат. порог):         {mde:.2f}%")
    print(f"Стат. значимость:          {'ДА ✓' if stat_sig else 'НЕТ (шум) ✗'}")
    print(f"{'─'*70}")
    print(f"Kelly (полный):            {kelly_result['kelly_full']*100:.1f}%")
    print(f"Kelly (×{KELLY_FRACTION}):           {kelly_result['kelly_fractional']*100:.1f}%")
    print(f"Kelly (рекомендация):      {kelly_result['kelly_capped']*100:.1f}% от капитала")
    print(f"Edge:                      {kelly_result.get('edge',0)*100:.3f}%")
    print(f"Валидность Келли:          {'✓' if kelly_result['is_valid'] else '✗'}")
    print(f"{'─'*70}")
    print(f"Средняя длительность:      {t_df['Duration Days'].mean():.1f} дней "
          f"({t_df['Duration Months'].mean():.2f} мес.)")
    print(f"{'═'*70}")
    
    return t_df, kelly_result

## Пайплайн запуска

In [ ]:
# === ПАЙПЛАЙН (оригинальный) ===
df_YDEX = pd.read_parquet('data/YDEX_5min.parquet')
df_YDEX = df_YDEX[df_YDEX['DateTime'] >= '2020-01-01']
df_YDEX = df_YDEX[df_YDEX['DateTime'] < '2026-01-01']
df_YDEX = fill_time_gaps(df_YDEX)
df_YDEX = clean_market_data(df_YDEX)

In [ ]:
# === ОБУЧЕНИЕ И БЭКТЕСТ ===
final_results = run_backtest_with_fine_tuning(df_YDEX)

In [ ]:
# === АНАЛИЗ С КЕЛЛИ ===
report_df, kelly_result = analyze_strategy_performance(final_results)
display(report_df.head(20))

# === ПОЛНЫЙ ОТЧЁТ КЕЛЛИ (все 3 формулировки) ===
kelly = KellyCriterion()
avg_hold = report_df['Duration Days'].mean() if 'Duration Days' in report_df.columns else 4.9
d, c, cr = kelly.full_report(report_df['Profit (доля)'].values, avg_holding_days=avg_hold, label="YDEX")

In [ ]:
def plot_trades(final_results, report_df, ticker: str = ""):
    """
    Визуализация сделок на ценовом графике.
    Поддерживает Long и Short позиции.
    
    - Зелёные ▲ — вход Long
    - Красные ▼ — вход Short
    - Синие ● — выход по сигналу (прибыль)
    - Серые ● — выход по сигналу (убыток)
    - Жёлтые ● — выход по Breakeven
    - Зелёная заливка — прибыльные Long
    - Оранжевая заливка — прибыльные Short
    - Красная заливка — убыточные позиции
    - Жёлтая заливка — breakeven
    """
    
    fig, axes = plt.subplots(2, 1, figsize=(18, 10), 
                              gridspec_kw={'height_ratios': [3, 1]}, sharex=True)
    
    # ── Верхний график: цена + сделки ──
    ax = axes[0]
    
    dates = pd.to_datetime(final_results['DateTime'])
    prices = final_results['Price'].values
    ax.plot(dates, prices, color='#555555', linewidth=0.8, alpha=0.7, label='Цена')
    
    for _, trade in report_df.iterrows():
        entry_date = pd.to_datetime(trade['Entry Date'])
        exit_date = pd.to_datetime(trade['Exit Date'])
        entry_price = trade['Entry Price']
        exit_price = trade['Exit Price']
        pnl = trade['Profit %']
        trade_type = trade['Type']
        direction = trade.get('Direction', 'Long')
        
        # Заливка
        if trade_type == 'Breakeven':
            fill_color, fill_alpha = '#FFD700', 0.08
        elif pnl > 0 and direction == 'Long':
            fill_color, fill_alpha = '#2ca02c', 0.12
        elif pnl > 0 and direction == 'Short':
            fill_color, fill_alpha = '#ff7f0e', 0.12
        else:
            fill_color, fill_alpha = '#d62728', 0.10
        
        ax.axvspan(entry_date, exit_date, color=fill_color, alpha=fill_alpha)
        
        # Вход
        if direction == 'Long':
            ax.scatter(entry_date, entry_price, marker='^', color='#2ca02c', 
                       s=100, zorder=5, edgecolors='black', linewidth=0.5)
        else:
            ax.scatter(entry_date, entry_price, marker='v', color='#d62728', 
                       s=100, zorder=5, edgecolors='black', linewidth=0.5)
        
        # Выход
        if trade_type == 'Breakeven':
            ax.scatter(exit_date, exit_price, marker='o', color='#FFD700', 
                       s=60, zorder=5, edgecolors='black', linewidth=0.5)
        elif pnl > 0:
            ax.scatter(exit_date, exit_price, marker='o', color='#1f77b4', 
                       s=80, zorder=5, edgecolors='black', linewidth=0.5)
        else:
            ax.scatter(exit_date, exit_price, marker='o', color='#999999', 
                       s=60, zorder=5, edgecolors='black', linewidth=0.5)
    
    # Легенда
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='^', color='w', markerfacecolor='#2ca02c', 
               markersize=10, markeredgecolor='black', label='Вход Long'),
        Line2D([0], [0], marker='v', color='w', markerfacecolor='#d62728', 
               markersize=10, markeredgecolor='black', label='Вход Short'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#1f77b4', 
               markersize=8, markeredgecolor='black', label='Выход (прибыль)'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#999999', 
               markersize=8, markeredgecolor='black', label='Выход (убыток)'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#FFD700', 
               markersize=8, markeredgecolor='black', label='Breakeven'),
    ]
    ax.legend(handles=legend_elements, loc='upper left', fontsize=9)
    ax.set_ylabel('Цена', fontsize=12)
    ax.set_title(f'Сделки на графике цены{" — " + ticker if ticker else ""}', fontsize=14)
    ax.grid(True, alpha=0.3)
    
    # ── Нижний график: equity curve ──
    ax2 = axes[1]
    
    sorted_trades = report_df.sort_values('Entry Date')
    cumulative_pnl = sorted_trades['Profit %'].cumsum()
    trade_dates = pd.to_datetime(sorted_trades['Exit Date'])
    
    colors = []
    for _, t in sorted_trades.iterrows():
        if t['Type'] == 'Breakeven':
            colors.append('#FFD700')
        elif t['Profit %'] > 0:
            colors.append('#2ca02c' if t.get('Direction','Long') == 'Long' else '#ff7f0e')
        else:
            colors.append('#d62728')
    
    ax2.plot(trade_dates.values, cumulative_pnl.values, color='#1f77b4', linewidth=2, label='Equity Curve')
    ax2.scatter(trade_dates.values, cumulative_pnl.values, c=colors, s=40, zorder=5, edgecolors='black', linewidth=0.3)
    ax2.axhline(y=0, color='black', linestyle='--', linewidth=0.5)
    ax2.fill_between(trade_dates.values, 0, cumulative_pnl.values, 
                      where=cumulative_pnl.values >= 0, color='#2ca02c', alpha=0.1)
    ax2.fill_between(trade_dates.values, 0, cumulative_pnl.values, 
                      where=cumulative_pnl.values < 0, color='#d62728', alpha=0.1)
    ax2.set_ylabel('Кумулятивный P&L (%)', fontsize=12)
    ax2.set_xlabel('Дата', fontsize=12)
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # ── Мини-статистика ──
    n_long = len(report_df[report_df.get('Direction', pd.Series()) == 'Long']) if 'Direction' in report_df.columns else len(report_df)
    n_short = len(report_df[report_df.get('Direction', pd.Series()) == 'Short']) if 'Direction' in report_df.columns else 0
    n_profit = len(report_df[report_df['Profit %'] > 0])
    n_loss = len(report_df[report_df['Profit %'] < 0])
    n_be = len(report_df[report_df['Type'] == 'Breakeven'])
    print(f"\nСделок: {len(report_df)} (Long: {n_long}, Short: {n_short})")
    print(f"Прибыльных: {n_profit}, Убыточных: {n_loss}, Breakeven: {n_be}")
    print(f"Лучшая:  {report_df['Profit %'].max():+.2f}%")
    print(f"Худшая:  {report_df['Profit %'].min():+.2f}%")
    
    return fig

In [ ]:
# === ВИЗУАЛИЗАЦИЯ СДЕЛОК ===
plot_trades(final_results, report_df, ticker="YDEX")

In [ ]:
# === СКОЛЬЗЯЩИЙ КЕЛЛИ (3 метода) ===
if report_df is not None and len(report_df) >= KELLY_MIN_TRADES:
    kelly = KellyCriterion()
    pnl = report_df['Profit (доля)'].values
    
    rolling_disc = kelly.rolling_kelly(pnl, method='discrete')
    rolling_cont = kelly.rolling_kelly(pnl, method='continuous')
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    
    # Верхний: Kelly capped (дискретный vs непрерывный)
    axes[0].plot(rolling_disc['trade_index'], rolling_disc['kelly_capped'] * 100, 
                 color='#1f77b4', linewidth=1.5, label=f'Дискретный ×{KELLY_FRACTION}')
    axes[0].plot(rolling_cont['trade_index'], rolling_cont['kelly_capped'] * 100, 
                 color='#ff7f0e', linewidth=1.5, alpha=0.8, label=f'Непрерывный ×{KELLY_FRACTION}')
    axes[0].axhline(y=0, color='red', linestyle='--', alpha=0.5)
    axes[0].set_ylabel('Размер позиции (%)')
    axes[0].set_title('Скользящий критерий Келли (дискретный vs непрерывный)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Нижний: Edge
    if 'edge' in rolling_disc.columns:
        axes[1].plot(rolling_disc['trade_index'], rolling_disc['edge'] * 100,
                     color='#2ca02c', linewidth=1.5, label='Edge (мат. ожидание)')
        axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)
        axes[1].set_ylabel('Edge (%)')
        axes[1].set_xlabel('Номер сделки')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # === АНАЛИЗ ЧУВСТВИТЕЛЬНОСТИ ===
    kelly.sensitivity_analysis(pnl)
    
    # === СРАВНЕНИЕ ФРАКЦИЙ ===
    kelly.compare_fractions(pnl)

else:
    print(f"Недостаточно сделок для анализа (нужно >= {KELLY_MIN_TRADES})")

## Сравнение

| Метрика | v2 (оригинал + Kelly) | v4 (+ недельный шаг) |
|---------|----------------------|---------------------|
| Сделок | 100 | — |
| Win Rate | 42.0% | — |
| Доходность | 9.42% | — |
| Ср. сделка | 0.09% | — |
| Hold period | 2.6 дн | — |
| PF | 1.20 | — |
| RR | 1.65 | — |
| Sharpe | 0.18 | — |
| MDE | 0.54% | — |
| Kelly | 0.6% | — |
| Edge | 0.034% | — |

Если v4 ≈ v2 → недельный шаг не влияет, можно добавлять следующее изменение.
Если v4 хуже → проблема в частоте fine-tuning, нужно искать другой рычаг.

## Пакетный запуск

In [ ]:
tickers_to_test = ['MGNT', 'VTBR', 'TATN', 'LKOH', 'YDEX', 'GLDRUB_TOM']

all_results = {}

for ticker in tickers_to_test:
    print(f"\n{'#'*70}")
    print(f"# {ticker}")
    print(f"{'#'*70}")
    
    try:
        df = pd.read_parquet(f'data/{ticker}_5min.parquet')
        df = df[df['DateTime'] < '2026-01-01']
        df = fill_time_gaps(df)
        df = clean_market_data(df)
        
        final_results = run_backtest_with_fine_tuning(df)
        result_df, kelly_res = analyze_strategy_performance(final_results)
        
        all_results[ticker] = {
            'trades': result_df,
            'kelly': kelly_res,
            'n_trades': len(result_df) if result_df is not None else 0
        }
        
    except Exception as e:
        print(f"Ошибка для {ticker}: {e}")
        all_results[ticker] = {'error': str(e)}